# 4.7 Coordinates and Change of Basis

**Course:** Elementary Linear Algebra (Larson, 8e)  
**Chapter 4:** Vector Spaces

---

### What you will learn

- How to express a vector as a **coordinate vector** $[\mathbf{x}]_B$ relative to a basis $B$.
- How to build the **transition matrix** $P$ that converts coordinates from one basis to another.
- Why $P^{-1}$ reverses the conversion, and how to verify the **roundtrip** property.
- How change of basis connects to **real-world coordinate systems** (e.g., world vs. camera coordinates).

---

## 1. Setup

In [ ]:
import sys, os
sys.path.insert(0, os.path.join(os.getcwd(), '../..'))

In [ ]:
from linalg_core.matrix import Matrix
from linalg_core.ops import multiply, inverse
from linalg_core.elimination import solve
from linalg_core.vecspace import coordinate_vector, change_of_basis
from linalg_core import EPSILON

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import math

print("Setup complete.")

---

## 2. Motivation --- World Coordinates to Camera Coordinates

A **graphics engine** stores every vertex of a 3D scene in *world coordinates*.
When the virtual camera moves or rotates, the engine must re-express every point
in *camera coordinates* --- a coordinate system whose origin is the camera
position and whose axes align with the camera's viewing direction.

This operation is a **change of basis**. The vertices themselves do not move;
only their *description* changes because we switch from one set of basis
vectors to another.

$$\text{world coords} \;\xrightarrow{\;P\;}\; \text{camera coords}
  \;\xrightarrow{\;P^{-1}\;}\; \text{world coords}$$

The matrix $P$ that performs this conversion is called the **transition matrix**
(or change-of-basis matrix). In this notebook we build, verify, and visualize
everything from scratch in $\mathbb{R}^2$, but the ideas extend directly to
$\mathbb{R}^3$ and beyond.

---

## 3. Build --- Core Concepts

### 3.1 Coordinate Vector

> **Definition (Larson, Sec. 4.7).** Let $B = \{\mathbf{v}_1, \mathbf{v}_2, \dots, \mathbf{v}_n\}$
> be a basis for a vector space $V$. For any $\mathbf{x} \in V$, there exist
> **unique** scalars $c_1, c_2, \dots, c_n$ such that
>
> $$\mathbf{x} = c_1 \mathbf{v}_1 + c_2 \mathbf{v}_2 + \cdots + c_n \mathbf{v}_n$$
>
> The vector $[\mathbf{x}]_B = \begin{bmatrix} c_1 \\ c_2 \\ \vdots \\ c_n \end{bmatrix}$
> is the **coordinate vector of $\mathbf{x}$ relative to $B$**.

In other words, the coordinate vector collects the weights needed to
reconstruct $\mathbf{x}$ from the basis vectors.

**How `coordinate_vector` works:** It solves the system
$[\mathbf{v}_1 \mid \mathbf{v}_2 \mid \cdots \mid \mathbf{v}_n \mid \mathbf{x}]$
for the unique weights $c_1, \dots, c_n$. If the vectors form a basis,
the solution is always unique.

In [ ]:
x = [6, 4]
std_basis = [[1, 0], [0, 1]]

coords_std = coordinate_vector(x, std_basis)

print("Vector x =", x)
print("Standard basis E = {e1, e2} =", std_basis)
print(f"\n[x]_E = {coords_std}")
print(f"\nReconstruction: {coords_std[0]}*e1 + {coords_std[1]}*e2"
      f" = [{coords_std[0]*1 + coords_std[1]*0}, {coords_std[0]*0 + coords_std[1]*1}]")
print("Matches x?", all(abs(a - b) < EPSILON for a, b in zip(x, [coords_std[0], coords_std[1]])))

### 3.2 Demo --- Custom Basis

Let $B = \left\{\begin{bmatrix}1\\1\end{bmatrix},\,\begin{bmatrix}1\\-1\end{bmatrix}\right\}$.

We seek $[\mathbf{x}]_B$ for $\mathbf{x} = \begin{bmatrix}3\\5\end{bmatrix}$.

We need scalars $c_1, c_2$ such that:

$$c_1 \begin{bmatrix}1\\1\end{bmatrix} + c_2 \begin{bmatrix}1\\-1\end{bmatrix}
  = \begin{bmatrix}3\\5\end{bmatrix}$$

By hand: $c_1 + c_2 = 3$ and $c_1 - c_2 = 5$, so $c_1 = 4$ and $c_2 = -1$.

$$[\mathbf{x}]_B = \begin{bmatrix}4\\-1\end{bmatrix}$$

In [ ]:
B = [[1, 1], [1, -1]]
x = [3, 5]

coords_B = coordinate_vector(x, B)

print(f"Basis B = {{v1={B[0]}, v2={B[1]}}}")
print(f"Vector x = {x}")
print(f"\n[x]_B = {coords_B}")
print(f"Expected: [4, -1]")

reconstructed = [
    coords_B[0] * B[0][0] + coords_B[1] * B[1][0],
    coords_B[0] * B[0][1] + coords_B[1] * B[1][1],
]
print(f"\nReconstruction: {coords_B[0]}*{B[0]} + {coords_B[1]}*{B[1]} = {reconstructed}")
print(f"Matches x? {all(abs(a - b) < EPSILON for a, b in zip(x, reconstructed))}")

### 3.3 Transition Matrix

> **Definition (Larson, Sec. 4.7).** Let $B$ and $B'$ be two bases for
> a vector space $V$. The **transition matrix from $B'$ to $B$** is the
> matrix $P$ such that
>
> $$[\mathbf{x}]_B = P \, [\mathbf{x}]_{B'}$$
>
> for every $\mathbf{x} \in V$.

The columns of $P$ are the coordinate vectors of the $B'$-basis vectors
expressed relative to $B$:

$$P = \bigl[ [\mathbf{v}'_1]_B \;\mid\; [\mathbf{v}'_2]_B \;\mid\; \cdots \;\mid\; [\mathbf{v}'_n]_B \bigr]$$

Our `change_of_basis(old_basis, new_basis)` function returns $P$ such that
$[\mathbf{x}]_{\text{new}} = P \, [\mathbf{x}]_{\text{old}}$.

In [ ]:
B = [[1, 1], [1, -1]]
B_prime = [[1, 0], [0, 1]]

P = change_of_basis(B_prime, B)

print("Old basis B' (standard) =", B_prime)
print("New basis B =", B)
print("\nTransition matrix P (from B' to B):")
print(P)

print("\nInterpretation:")
print(f"  Column 0 = [e1]_B = coordinate_vector({B_prime[0]}, B) = {coordinate_vector(B_prime[0], B)}")
print(f"  Column 1 = [e2]_B = coordinate_vector({B_prime[1]}, B) = {coordinate_vector(B_prime[1], B)}")

### 3.4 Roundtrip --- Standard to $B'$ and Back

A key property of the transition matrix is that it is **invertible**.
If $P$ converts coordinates from $B'$ to $B$, then $P^{-1}$ converts
from $B$ back to $B'$.

Let us verify the roundtrip:

$$\mathbf{x} \;\xrightarrow{\text{find } [\mathbf{x}]_{B'}}\;
  [\mathbf{x}]_{B'} \;\xrightarrow{\;P\;}\;
  [\mathbf{x}]_B \;\xrightarrow{\;P^{-1}\;}\;
  [\mathbf{x}]_{B'}$$

We start with $\mathbf{x}$ in standard coordinates, convert to $B$, then
convert back and check that we recover the original coordinates.

In [ ]:
B = [[1, 1], [1, -1]]
B_prime = [[1, 0], [0, 1]]
x = [3, 5]

coords_Bp = coordinate_vector(x, B_prime)
print(f"Step 1: [x]_B' = {coords_Bp}")
print(f"  (In standard coords, this is just x itself.)")

P = change_of_basis(B_prime, B)
coords_Bp_col = Matrix.from_vector(coords_Bp)
coords_B_col = multiply(P, coords_Bp_col)
coords_B = coords_B_col.get_col(0)
print(f"\nStep 2: [x]_B = P * [x]_B' = {coords_B}")

P_inv = inverse(P)
recovered_col = multiply(P_inv, coords_B_col)
recovered = recovered_col.get_col(0)
print(f"\nStep 3: P^(-1) * [x]_B = {recovered}")

match = all(abs(a - b) < EPSILON for a, b in zip(coords_Bp, recovered))
print(f"\nRoundtrip recovery matches? {match}")

### 3.5 Inverse Transition --- $P^{-1}$ Converts the Other Direction

> **Theorem (Larson, Sec. 4.7).** If $P$ is the transition matrix from $B'$
> to $B$, then $P$ is invertible and $P^{-1}$ is the transition matrix from
> $B$ to $B'$.

This means:

$$P \cdot P^{-1} = I \quad \text{and} \quad P^{-1} \cdot P = I$$

Let us verify both products equal the identity matrix.

In [ ]:
B = [[1, 1], [1, -1]]
B_prime = [[1, 0], [0, 1]]

P = change_of_basis(B_prime, B)
P_inv = inverse(P)
I2 = Matrix.identity(2)

print("P (B' -> B):")
print(P)
print("\nP^(-1) (B -> B'):")
print(P_inv)

product_1 = multiply(P, P_inv)
product_2 = multiply(P_inv, P)

print("\nP * P^(-1):")
print(product_1)
print(f"Equals I? {product_1 == I2}")

print("\nP^(-1) * P:")
print(product_2)
print(f"Equals I? {product_2 == I2}")

Q = change_of_basis(B, B_prime)
print("\nDirect computation: transition matrix from B to B':")
print(Q)
print(f"Q == P^(-1)? {Q == P_inv}")

### 3.6 Camera Analogy --- 2D World and Camera Bases

Imagine a 2D game where the **world basis** is the standard basis
$E = \{\mathbf{e}_1, \mathbf{e}_2\}$ and the **camera basis** represents
a camera rotated $45^\circ$ counterclockwise:

$$C = \left\{\begin{bmatrix}\cos 45^\circ \\ \sin 45^\circ\end{bmatrix},\;
  \begin{bmatrix}-\sin 45^\circ \\ \cos 45^\circ\end{bmatrix}\right\}
  = \left\{\begin{bmatrix}\tfrac{\sqrt{2}}{2} \\ \tfrac{\sqrt{2}}{2}\end{bmatrix},\;
  \begin{bmatrix}-\tfrac{\sqrt{2}}{2} \\ \tfrac{\sqrt{2}}{2}\end{bmatrix}\right\}$$

We convert several points from world coordinates to camera coordinates.

In [ ]:
c45 = math.cos(math.pi / 4)
s45 = math.sin(math.pi / 4)

world_basis = [[1, 0], [0, 1]]
camera_basis = [[c45, s45], [-s45, c45]]

P_world_to_cam = change_of_basis(world_basis, camera_basis)

print("World basis (standard):")
for i, v in enumerate(world_basis):
    print(f"  e{i+1} = {v}")

print(f"\nCamera basis (rotated 45 deg):")
for i, v in enumerate(camera_basis):
    print(f"  c{i+1} = [{v[0]:.4f}, {v[1]:.4f}]")

print("\nTransition matrix P (world -> camera):")
print(P_world_to_cam)

test_points = [[1, 0], [0, 1], [3, 4], [1, 1]]

print("\n" + "-" * 50)
print(f"{'World coords':<20} {'Camera coords':<20}")
print("-" * 50)

for pt in test_points:
    world_col = Matrix.from_vector(pt)
    cam_col = multiply(P_world_to_cam, world_col)
    cam_coords = cam_col.get_col(0)
    print(f"{str(pt):<20} [{cam_coords[0]:>7.4f}, {cam_coords[1]:>7.4f}]")

print("\nNote: [1,1] in world coords lies exactly along the camera's")
print("first axis, so its camera coords are [sqrt(2), 0].")
print(f"sqrt(2) = {math.sqrt(2):.4f}")

---

## 4. Verify --- Cross-Check with NumPy

We verify our from-scratch coordinate vectors and transition matrices
against NumPy's linear algebra routines.

In [ ]:
def to_np(mat):
    """Convert our Matrix to a NumPy array."""
    return np.array(mat.to_lists())


def check_match(ours, theirs, label):
    """Compare our result against a NumPy array."""
    if isinstance(ours, list):
        ours_np = np.array(ours)
    else:
        ours_np = to_np(ours)
    match = np.allclose(ours_np, theirs, atol=1e-9)
    status = "PASS" if match else "FAIL"
    print(f"[{status}] {label}")
    if not match:
        print(f"  Ours:  {ours_np}")
        print(f"  NumPy: {theirs}")
    return match

In [ ]:
B_np = np.array([[1, 1], [1, -1]]).T
x_np = np.array([3, 5])

coords_np = np.linalg.solve(B_np, x_np)
coords_ours = coordinate_vector([3, 5], [[1, 1], [1, -1]])
check_match(coords_ours, coords_np, "coordinate_vector([3,5], B) matches np.linalg.solve")

B_lists = [[1, 1], [1, -1]]
Bp_lists = [[1, 0], [0, 1]]
P_ours = change_of_basis(Bp_lists, B_lists)

Bp_np = np.array(Bp_lists).T
B_mat_np = np.array(B_lists).T
P_np = np.linalg.solve(B_mat_np, Bp_np)
check_match(P_ours, P_np, "change_of_basis(B', B) matches NumPy transition matrix")

P_inv_ours = to_np(inverse(P_ours))
P_inv_np = np.linalg.inv(P_np)
check_match_inv = np.allclose(P_inv_ours, P_inv_np, atol=1e-9)
print(f"[{'PASS' if check_match_inv else 'FAIL'}] inverse(P) matches np.linalg.inv(P)")

roundtrip_np = P_inv_np @ P_np
identity_match = np.allclose(roundtrip_np, np.eye(2), atol=1e-9)
print(f"[{'PASS' if identity_match else 'FAIL'}] P^(-1) * P == I (NumPy roundtrip)")

In [ ]:
print("=" * 55)
print("VERIFICATION: Camera basis roundtrip")
print("=" * 55)

cam_np = np.array(camera_basis).T
P_cam_np = np.linalg.solve(cam_np, np.eye(2))
check_match(P_world_to_cam, P_cam_np, "World->Camera transition matrix matches NumPy")

for pt in test_points:
    pt_np = np.array(pt, dtype=float)
    cam_np_result = P_cam_np @ pt_np

    world_col = Matrix.from_vector(pt)
    cam_col = multiply(P_world_to_cam, world_col)
    cam_ours = cam_col.get_col(0)

    ok = np.allclose(cam_ours, cam_np_result, atol=1e-9)
    print(f"[{'PASS' if ok else 'FAIL'}] Point {pt}: camera coords match")

---

## 5. Visualize --- Two Overlaid Coordinate Grids

We overlay the **standard grid** (gray) with a **custom basis grid** (blue)
and show a point with its coordinates in both systems.

This makes the geometric meaning of "change of basis" visible:
the same point in the plane has different numerical coordinates
depending on which grid you read off.

In [ ]:
B = [[1, 1], [1, -1]]
v1 = np.array(B[0])
v2 = np.array(B[1])
x = np.array([3.0, 5.0])
coords_B = coordinate_vector(x.tolist(), B)

fig, ax = plt.subplots(1, 1, figsize=(9, 9))

grid_range = range(-4, 6)
for i in grid_range:
    ax.axhline(i, color='gray', linewidth=0.3, alpha=0.5)
    ax.axvline(i, color='gray', linewidth=0.3, alpha=0.5)
ax.axhline(0, color='gray', linewidth=1.0, alpha=0.7)
ax.axvline(0, color='gray', linewidth=1.0, alpha=0.7)

for i in grid_range:
    start_1 = i * v1 + min(grid_range) * v2
    end_1 = i * v1 + max(grid_range) * v2
    ax.plot([start_1[0], end_1[0]], [start_1[1], end_1[1]],
            color='steelblue', linewidth=0.5, alpha=0.4)

    start_2 = min(grid_range) * v1 + i * v2
    end_2 = max(grid_range) * v1 + i * v2
    ax.plot([start_2[0], end_2[0]], [start_2[1], end_2[1]],
            color='steelblue', linewidth=0.5, alpha=0.4)

ax.annotate('', xy=v1, xytext=(0, 0),
            arrowprops=dict(arrowstyle='->', color='steelblue', lw=2.0))
ax.text(v1[0] + 0.1, v1[1] + 0.15, r'$\mathbf{v}_1$',
        fontsize=14, color='steelblue', fontweight='bold')

ax.annotate('', xy=v2, xytext=(0, 0),
            arrowprops=dict(arrowstyle='->', color='royalblue', lw=2.0))
ax.text(v2[0] + 0.1, v2[1] - 0.3, r'$\mathbf{v}_2$',
        fontsize=14, color='royalblue', fontweight='bold')

ax.annotate('', xy=(1, 0), xytext=(0, 0),
            arrowprops=dict(arrowstyle='->', color='dimgray', lw=1.5))
ax.text(1.05, -0.25, r'$\mathbf{e}_1$', fontsize=12, color='dimgray')

ax.annotate('', xy=(0, 1), xytext=(0, 0),
            arrowprops=dict(arrowstyle='->', color='dimgray', lw=1.5))
ax.text(-0.35, 1.05, r'$\mathbf{e}_2$', fontsize=12, color='dimgray')

ax.plot(x[0], x[1], 'o', color='crimson', markersize=10, zorder=5)
ax.annotate('', xy=x, xytext=(0, 0),
            arrowprops=dict(arrowstyle='->', color='crimson', lw=2.0))

label_std = f'Standard: $[\\mathbf{{x}}]_E = ({x[0]:.0f},\, {x[1]:.0f})$'
label_B = (f'Basis $B$: $[\\mathbf{{x}}]_B = '
           f'({coords_B[0]:.0f},\, {coords_B[1]:.0f})$')

ax.text(x[0] + 0.2, x[1] + 0.35, label_std,
        fontsize=11, color='dimgray',
        bbox=dict(boxstyle='round,pad=0.3', facecolor='white', alpha=0.9))
ax.text(x[0] + 0.2, x[1] - 0.45, label_B,
        fontsize=11, color='steelblue',
        bbox=dict(boxstyle='round,pad=0.3', facecolor='aliceblue', alpha=0.9))

p1 = coords_B[0] * v1
p2 = p1 + coords_B[1] * v2
ax.plot([0, p1[0]], [0, p1[1]], '--', color='steelblue', linewidth=1.2, alpha=0.6)
ax.plot([p1[0], p2[0]], [p1[1], p2[1]], '--', color='royalblue', linewidth=1.2, alpha=0.6)

ax.text(p1[0] / 2 - 0.4, p1[1] / 2 + 0.15,
        f'{coords_B[0]:.0f}$\\mathbf{{v}}_1$',
        fontsize=10, color='steelblue', alpha=0.8)
ax.text((p1[0] + p2[0]) / 2 + 0.15, (p1[1] + p2[1]) / 2 - 0.1,
        f'{coords_B[1]:.0f}$\\mathbf{{v}}_2$',
        fontsize=10, color='royalblue', alpha=0.8)

import matplotlib.lines as mlines
gray_line = mlines.Line2D([], [], color='gray', linewidth=0.8, label='Standard grid')
blue_line = mlines.Line2D([], [], color='steelblue', linewidth=0.8, label='Basis $B$ grid')
red_dot = mlines.Line2D([], [], color='crimson', marker='o', linestyle='None',
                         markersize=8, label=r'$\mathbf{x}$')
ax.legend(handles=[gray_line, blue_line, red_dot], loc='lower right', fontsize=11)

ax.set_xlim(-3, 7)
ax.set_ylim(-3, 7)
ax.set_aspect('equal')
ax.set_title('Two Coordinate Systems: Standard (gray) vs. Basis $B$ (blue)',
             fontsize=13, fontweight='bold', pad=12)
ax.set_xlabel('$x_1$', fontsize=12)
ax.set_ylabel('$x_2$', fontsize=12)

plt.tight_layout()
plt.show()

---

## 6. Exercises

Test your understanding with the following problems. Write your solutions
in the code cells provided.

### Exercise 1 (Easy)

Let $B = \left\{\begin{bmatrix}2\\1\end{bmatrix},\,\begin{bmatrix}1\\3\end{bmatrix}\right\}$.

Find the coordinate vector $[\mathbf{x}]_B$ for $\mathbf{x} = \begin{bmatrix}5\\3\end{bmatrix}$.

**By hand:** Solve
$c_1 \begin{bmatrix}2\\1\end{bmatrix} + c_2 \begin{bmatrix}1\\3\end{bmatrix}
= \begin{bmatrix}5\\3\end{bmatrix}$,
i.e., $2c_1 + c_2 = 5$ and $c_1 + 3c_2 = 3$.

Then verify with `coordinate_vector`.

In [ ]:
# YOUR CODE HERE


### Exercise 2 (Medium)

Let $B' = \{\mathbf{e}_1, \mathbf{e}_2\}$ (standard basis) and
$B = \left\{\begin{bmatrix}2\\1\end{bmatrix},\,\begin{bmatrix}1\\3\end{bmatrix}\right\}$.

1. Compute the transition matrix $P$ from standard to $B$ using `change_of_basis`.
2. Use $P$ to convert $\mathbf{x} = \begin{bmatrix}7\\11\end{bmatrix}$ from standard to $B$-coordinates.
3. Verify by computing `coordinate_vector(x, B)` directly.
4. Check that $P^{-1} P = I$.

In [ ]:
# YOUR CODE HERE


### Exercise 3 (Challenge)

In $\mathbb{R}^3$, let

$$B = \left\{\begin{bmatrix}1\\0\\1\end{bmatrix},\,
  \begin{bmatrix}0\\1\\1\end{bmatrix},\,
  \begin{bmatrix}1\\1\\0\end{bmatrix}\right\},
  \quad
  B' = \left\{\begin{bmatrix}1\\0\\0\end{bmatrix},\,
  \begin{bmatrix}0\\1\\0\end{bmatrix},\,
  \begin{bmatrix}0\\0\\1\end{bmatrix}\right\}$$

1. Compute the transition matrix $P$ directly from $B$ to $B'$ (i.e., convert
   $B$-coordinates to standard coordinates).
2. Compute $P^{-1}$ (the reverse: standard to $B$).
3. Pick an arbitrary $\mathbf{x} \in \mathbb{R}^3$ and verify the full roundtrip:
   $P^{-1} \bigl( P \, [\mathbf{x}]_B \bigr) = [\mathbf{x}]_B$.
4. **Bonus:** Compute the transition matrix directly from $B$ to a third basis
   $B'' = \{[2,1,0],\,[0,2,1],\,[1,0,2]\}$ without going through the standard
   basis as an intermediate step.

In [ ]:
# YOUR CODE HERE


---

## Summary

| Concept | Key Takeaway |
|---|---|
| **Coordinate vector** $[\mathbf{x}]_B$ | The unique scalars $c_1, \dots, c_n$ such that $\mathbf{x} = c_1\mathbf{v}_1 + \cdots + c_n\mathbf{v}_n$ |
| **Transition matrix** $P$ | Converts coordinates from one basis to another: $[\mathbf{x}]_B = P\,[\mathbf{x}]_{B'}$ |
| **Columns of $P$** | Each column $j$ is $[\mathbf{v}'_j]_B$ --- the $j$-th old-basis vector written in new-basis coordinates |
| **Invertibility** | $P$ is always invertible; $P^{-1}$ reverses the conversion |
| **Roundtrip** | $P^{-1}(P\,[\mathbf{x}]_{B'}) = [\mathbf{x}]_{B'}$ --- convert and convert back recovers the original |
| **Application** | Graphics engines use change of basis to switch between world, camera, and screen coordinates |